# 06 · Model Building
**Project:** EV Charging Demand & Cost Analytics + Predictive System  
**Input:** `data/processed/ev_charging_features.csv`  
**Output:** Trained models saved to `src/models/`

---

### Goals
1. **Track A (Regression):** Predict `Charging Cost (USD)`.
2. **Track B (Classification):** Predict `User Type`.
3. **Preprocessing:** Handle ID columns, temporal leakage, and encode categorical features. Scale numeric features.
4. **Modeling:** Train and compare Linear/Logistic Regression, Random Forest, and XGBoost.
5. **Export:** Save the best models for future deployment.

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings('ignore')

# Ensure model directory exists
os.makedirs('../src/models', exist_ok=True)

FEATURES_PATH = '../data/processed/ev_charging_features.csv'
df = pd.read_csv(FEATURES_PATH)

print(f'Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')

Loaded dataset: 1,320 rows × 30 columns


---
## 1. Global Preprocessing Strategy
We need to drop columns that won't generalize or are raw IDs. We will also define our numeric and categorical columns.

In [2]:
# Columns to drop completely (IDs and raw datetime strings)
cols_to_drop = ['User ID', 'Charging Station ID', 'Charging Start Time', 'Charging End Time']
df = df.drop(columns=cols_to_drop)

print(f"Remaining columns: {df.shape[1]}")

Remaining columns: 26


---
## Track A: Regression (Predicting Charging Cost)

In [3]:
target_A = 'Charging Cost (USD)'

# Data Leakage Columns: cost_per_kwh is directly derived from Charging Cost
leakage_cols_A = ['cost_per_kwh', 'charging_efficiency', 'distance_per_kwh', 'Charging Duration (hours)', 'Energy Consumed (kWh)', 'State of Charge (End %)', 'soc_gained_pct', 'battery_utilization_ratio']

X_A = df.drop(columns=[target_A] + leakage_cols_A)
y_A = df[target_A]

# Identify column types for Pipeline
cat_cols_A = X_A.select_dtypes(include=['object']).columns.tolist()
num_cols_A = X_A.select_dtypes(include=[np.number]).columns.tolist()

print(f"Track A Features: {len(X_A.columns)} ({len(cat_cols_A)} categorical, {len(num_cols_A)} numeric)")

# Train-test split (80/20)
X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(X_A, y_A, test_size=0.2, random_state=42)

# Preprocessing pipeline
preprocessor_A = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols_A),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols_A)
    ])

# Models to test
models_A = {
    'Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(random_state=42),
    'XGBoost Regressor': XGBRegressor(random_state=42)
}

results_A = []
best_model_A = None
best_r2 = -float('inf')
best_model_name_A = ""

for name, model in models_A.items():
    # Create full pipeline
    pipeline = Pipeline(steps=[('preprocessor', preprocessor_A),
                               ('model', model)])
    # Train
    pipeline.fit(X_train_A, y_train_A)
    
    # Predict
    y_pred = pipeline.predict(X_test_A)
    
    # Evaluate
    rmse = np.sqrt(mean_squared_error(y_test_A, y_pred))
    mae = mean_absolute_error(y_test_A, y_pred)
    r2 = r2_score(y_test_A, y_pred)
    
    results_A.append({
        'Model': name,
        'RMSE': rmse,
        'MAE': mae,
        'R2 Score': r2
    })
    
    if r2 > best_r2:
        best_r2 = r2
        best_model_A = pipeline
        best_model_name_A = name

res_df_A = pd.DataFrame(results_A).sort_values(by='R2 Score', ascending=False)
print("\n=== Track A Results (Regression on Charging Cost) ===")
display(res_df_A)

# Save best model
model_path_A = '../src/models/best_cost_regressor.pkl'
joblib.dump(best_model_A, model_path_A)
print(f"Saved Best Model ({best_model_name_A}) to {model_path_A}")

Track A Features: 17 (6 categorical, 11 numeric)



=== Track A Results (Regression on Charging Cost) ===


,Model,RMSE,MAE,R2 Score
0,Linear Regression,11.160322,9.379877,0.006799
1,Random Forest Regressor,11.271770,9.451712,-0.013137
2,XGBoost Regressor,11.906368,9.601790,-0.130427


Saved Best Model (Linear Regression) to ../src/models/best_cost_regressor.pkl


---
## Track B: Classification (Predicting User Type)

In [4]:
target_B = 'User Type'

X_B = df.drop(columns=[target_B, 'Charging Cost (USD)', 'cost_per_kwh', 'charging_efficiency', 'distance_per_kwh', 'Charging Duration (hours)', 'Energy Consumed (kWh)', 'State of Charge (End %)', 'soc_gained_pct', 'battery_utilization_ratio'])
y_B = df[target_B]

# Convert target to numerical for XGBoost
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_B_encoded = le.fit_transform(y_B)

# Identify column types for Pipeline
cat_cols_B = X_B.select_dtypes(include=['object']).columns.tolist()
num_cols_B = X_B.select_dtypes(include=[np.number]).columns.tolist()

print(f"Track B Features: {len(X_B.columns)} ({len(cat_cols_B)} categorical, {len(num_cols_B)} numeric)")

# Train-test split (80/20)
X_train_B, X_test_B, y_train_B, y_test_B = train_test_split(X_B, y_B_encoded, test_size=0.2, random_state=42)

# Preprocessing pipeline
preprocessor_B = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols_B),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols_B)
    ])

# Models to test
models_B = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest Classifier': RandomForestClassifier(random_state=42),
    'XGBoost Classifier': XGBClassifier(random_state=42, eval_metric='mlogloss')
}

results_B = []
best_model_B = None
best_f1 = -float('inf')
best_model_name_B = ""

for name, model in models_B.items():
    # Create full pipeline
    pipeline = Pipeline(steps=[('preprocessor', preprocessor_B),
                               ('model', model)])
    # Train
    pipeline.fit(X_train_B, y_train_B)
    
    # Predict
    y_pred = pipeline.predict(X_test_B)
    
    # Evaluate
    acc = accuracy_score(y_test_B, y_pred)
    prec = precision_score(y_test_B, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test_B, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test_B, y_pred, average='weighted', zero_division=0)
    
    results_B.append({
        'Model': name,
        'Accuracy': acc,
        'Precision (Weighted)': prec,
        'Recall (Weighted)': rec,
        'F1 Score (Weighted)': f1
    })
    
    if f1 > best_f1:
        best_f1 = f1
        best_model_B = pipeline
        best_model_name_B = name

res_df_B = pd.DataFrame(results_B).sort_values(by='F1 Score (Weighted)', ascending=False)
print("\n=== Track B Results (Classification on User Type) ===")
display(res_df_B)

# Save best model & label encoder (needed for decoding predictions later)
model_path_B = '../src/models/best_user_classifier.pkl'
le_path = '../src/models/user_type_encoder.pkl'
joblib.dump(best_model_B, model_path_B)
joblib.dump(le, le_path)
print(f"Saved Best Model ({best_model_name_B}) to {model_path_B}")
print(f"Saved LabelEncoder to {le_path}")

Track B Features: 16 (5 categorical, 11 numeric)



=== Track B Results (Classification on User Type) ===


,Model,Accuracy,Precision (Weighted),Recall (Weighted),F1 Score (Weighted)
2,XGBoost Classifier,0.378788,0.393249,0.378788,0.380726
0,Logistic Regression,0.352273,0.350646,0.352273,0.348841
1,Random Forest Classifier,0.337121,0.330429,0.337121,0.327172


Saved Best Model (XGBoost Classifier) to ../src/models/best_user_classifier.pkl
Saved LabelEncoder to ../src/models/user_type_encoder.pkl


---
## Summary
Models for both tracks have been trained, evaluated, and the top performers successfully exported. These pipelines (including their preprocessing logic) can now be directly loaded in `app.py` for serving predictions.